# 02 · 把高斯涂到像素上

上一篇找到了高斯在照片上的中心和形状。现在把照片放大到像素级：**中心附近应该多亮，边缘应该多淡，两团颜色重叠时谁盖住谁？**

我们先画一个柔和的光斑，再叠两层彩色玻璃，最后交换顺序，亲眼看看颜色怎样变化。

使用本主题环境从头运行。场景采用原点相机、单位旋转，所有高斯中心都在相机前方；这样可以把注意力集中到像素着色上。

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

接下来用两种画法：第一节直接累加一个高斯的颜色贡献；第二节加入透明度覆盖，让前面的颜色遮住后面的一部分。

## 1. 高斯像一座坡度柔和的小山

把二维高斯看成一座山包：中心是山顶，越往外走，高度越低。我们把这个“高度”当作像素权重：

$$g=\exp\left(-\frac12\Delta^T\Sigma^{-1}\Delta\right),\qquad
\Delta=p-\mu.$$

在中心，Δ=0，权重就是 1；离中心越远，权重逐渐衰减。协方差决定山包在哪个方向更宽，逆协方差则把像素距离换成“相对这个方向有多远”。

再乘上整体不透明度，就得到这个像素的 alpha：$\alpha=\text{opacity}\cdot g$。

### 给小山圈一道围栏

高斯的尾巴延伸很远。为了省计算，下面只遍历中心附近的 3σ 包围盒，并跳过 alpha ≤ 0.05 的像素。这两个设置决定了光斑边缘截到哪里。

<details>
<summary>包围盒和逆矩阵怎么算？</summary>

若 $\Sigma=\begin{bmatrix}a&b\\b&c\end{bmatrix}$，横纵半径分别取 $3\sqrt a$、$3\sqrt c$；a、c 是两个坐标方向的边缘方差。

逆矩阵为 $\Sigma^{-1}=\frac1{ac-b^2}\begin{bmatrix}c&-b\\-b&a\end{bmatrix}$。代码在行列式过小时跳过该足迹，避免不稳定的求逆。

</details>

### 实现：逐个像素问“你离山顶有多远？”

先算逆协方差和包围盒，再遍历盒内像素。每个像素得到一个权重，乘上 opacity 和 RGB，累加到画布上。

下面画一个倾斜的橙色光斑，看看它的亮度如何从中心向外过渡。

In [ ]:
def compute_inverted_covariance(cov2d):
    """
    计算 2D 协方差矩阵的逆矩阵
    cov2d: (2, 2)
    """
    # 1. 计算行列式 det = ad - bc
    det = np.linalg.det(cov2d)

    # 保护机制：如果行列式太小（矩阵奇异），则无法求逆
    if det < 1e-6:
        return None

    # 2. 计算逆矩阵
    # inv = [[d, -b], [-c, a]] / det
    inv_cov = np.linalg.inv(cov2d)
    return inv_cov


def get_bounding_box(center_u, center_v, cov2d, image_width, image_height):
    """
    计算 2D 高斯的包围盒 (3倍标准差)
    """
    # 提取对角线元素 (方差)
    var_x = cov2d[0, 0]
    var_y = cov2d[1, 1]

    # 标准差 = sqrt(方差)
    radius_x = 3.0 * math.sqrt(var_x)
    radius_y = 3.0 * math.sqrt(var_y)

    # 计算边界 (取整)
    min_x = max(0, int(center_u - radius_x))
    max_x = min(image_width, int(center_u + radius_x) + 1)

    min_y = max(0, int(center_v - radius_y))
    max_y = min(image_height, int(center_v + radius_y) + 1)

    return min_x, max_x, min_y, max_y


def rasterize_single_gaussian(image, center_u, center_v, cov2d, color, opacity):
    """
    光栅化单个高斯到图像上 (Additive Blending)

    参数:
    image: (H, W, 3) 画布
    center_u, center_v: 高斯中心坐标
    cov2d: (2, 2) 协方差矩阵
    color: (3,) RGB 颜色
    opacity: 不透明度 (0.0 - 1.0)
    """
    H, W, _ = image.shape

    # 1. 计算逆协方差矩阵
    inv_cov = compute_inverted_covariance(cov2d)
    if inv_cov is None:
        return # 忽略奇异矩阵

    # 2. 计算包围盒
    min_x, max_x, min_y, max_y = get_bounding_box(center_u, center_v, cov2d, W, H)

    # 3. 遍历包围盒内的像素
    for y in range(min_y, max_y):
        for x in range(min_x, max_x):
            # 像素坐标相对于高斯中心的偏移
            delta = np.array([x - center_u, y - center_v])
            # 公式: power = exp(-0.5 * delta.T @ inv_cov @ delta)
            power = math.exp(-0.5 * (delta @ inv_cov @ delta))

            # 最终 Alpha = 基础不透明度 * 高斯强度
            alpha = opacity * power

            # 简单的叠加混合 (Additive Blending) - 仅用于演示
            # 实际 3DGS 使用的是 Alpha Blending (从前向后或从后向前)
            if alpha > 0.05: # 忽略太淡的像素
                image[y, x] += color * alpha

    return image

In [ ]:
width, height = 96, 80
canvas = np.zeros((height, width, 3))
covariance = np.array([[100.0, 55.0], [55.0, 80.0]])
rasterize_single_gaussian(canvas, 48.0, 40.0, covariance, np.array([1.0, 0.3, 0.0]), 0.8)
plt.figure(figsize=(6, 4))
plt.imshow(np.clip(canvas, 0, 1))
plt.title("Single Gaussian footprint")
plt.xlabel("u")
plt.ylabel("v")
plt.show()

中心颜色最强，沿长轴方向衰减得更慢。若边缘出现明显截断，可以回头看 3σ 包围盒和 0.05 阈值：它们决定了哪些微弱贡献被省略。

## 2. 叠彩色玻璃：为什么先后顺序会改变颜色？

先在黑底上放一片 alpha=0.5 的纯蓝玻璃，再放一片 alpha=0.5 的纯红玻璃。红色贡献 0.5，已有蓝色则只剩一半，最终是“0.5 红 + 0.25 蓝”。交换顺序，就成了“0.25 红 + 0.5 蓝”。

这正是从远到近绘制时的更新公式：

$$C_{new}=\alpha C_{src}+(1-\alpha)C_{old}.$$

**新颜色拿走 α 的份额，旧画面保留 1−α。** 像画家先画远景，再往上叠近景，这种做法叫画家算法。

下面把一红一蓝两个高斯放在不同深度，并调整尺度，让它们在屏幕上覆盖相近的区域。

<details>
<summary>从近到远累积，又怎样算？</summary>

前面每层透过的比例是 $1-\alpha$，到第 i 层时剩余的透射率为 $T_i=\prod_{j<i}(1-\alpha_j)$。于是黑背景上的颜色为 $C=\sum_i T_i\alpha_i c_i$；有背景色时再加剩余透射率乘背景色。

这种前向累积可以在 T 足够小时提前结束。本篇使用上面的从远到近覆盖公式，排序依据是相机空间的 z 深度。

</details>

### 先把投影工具准备好

这里再次定义旋转、三维协方差和二维投影函数，让本篇能从新内核直接开始。它们负责把场景中的高斯变成像素循环需要的中心和形状。

In [ ]:
def get_rotation_matrix_from_euler(x, y, z):
    """
    简单的辅助函数：将欧拉角(弧度)转换为旋转矩阵 R
    这里为了简单直观，先使用欧拉角，实际项目中通常使用四元数(Quaternion)
    """
    R_x = np.array([[1, 0, 0],
                    [0, math.cos(x), -math.sin(x)],
                    [0, math.sin(x), math.cos(x)]])

    R_y = np.array([[math.cos(y), 0, math.sin(y)],
                    [0, 1, 0],
                    [-math.sin(y), 0, math.cos(y)]])

    R_z = np.array([[math.cos(z), -math.sin(z), 0],
                    [math.sin(z), math.cos(z), 0],
                    [0, 0, 1]])

    R = np.dot(R_z, np.dot(R_y, R_x))
    return R


def compute_covariance_3d(scale, rotation_matrix):
    """
    计算 3D 协方差矩阵 Sigma

    参数:
    scale: (3,) numpy array, 代表 x, y, z 三个主轴的标准差
    rotation_matrix: (3, 3) numpy array, 旋转矩阵 R

    返回:
    Sigma: (3, 3) 协方差矩阵
    """
    # 1. 构建缩放矩阵 S (对角矩阵)
    # S 是一个 3x3 矩阵，对角线上是 scale 的值，其他地方是 0
    S = np.diag(scale)
    R = rotation_matrix

    # 2. 计算协方差矩阵 Sigma = R * S * S^T * R^T
    # 提示：在 numpy 中，矩阵乘法可以使用 @ 符号，转置使用 .T
    Sigma = R @ S @ S.T @ R.T

    return Sigma


def compute_2d_covariance(covariance_3d, view_matrix, focal_x, focal_y, view_space_pos):
    """
    计算 2D 投影协方差矩阵 (EWA Splatting)

    参数:
    covariance_3d: (3, 3) 3D 协方差矩阵
    view_matrix: (3, 3) 视图矩阵的旋转部分 (World to Camera Rotation)
    focal_x, focal_y: 相机焦距
    view_space_pos: (3,) 高斯中心在相机坐标系下的位置 (x, y, z)

    返回:
    cov2d: (2, 2) 2D 协方差矩阵
    """
    # 1. 提取相机坐标系下的坐标
    x, y, z = view_space_pos

    # 2. 构建雅可比矩阵 J (2x3)
    # J 描述了 3D 位置微小变化如何影响 2D 屏幕坐标
    # 公式:
    # J = [ fx/z   0    -(fx*x)/(z*z) ]
    #     [ 0    fy/z   -(fy*y)/(z*z) ]
    J = np.zeros((2, 3))
    # 第一行：u 对 x, y, z 的偏导数
    J[0, 0] = focal_x / z
    J[0, 1] = 0
    J[0, 2] = -(focal_x * x) / (z * z)

    # 第二行：v 对 x, y, z 的偏导数
    J[1, 0] = 0
    J[1, 1] = focal_y / z
    J[1, 2] = -(focal_y * y) / (z * z)

    # 3. 计算变换矩阵 T = J @ W
    # W 是 view_matrix (3x3)
    W = view_matrix
    T = J @ W

    # 4. 计算 2D 协方差: Sigma_2D = T @ Sigma_3D @ T.T
    cov2d = T @ covariance_3d @ T.T

    return cov2d

### 实现：先画远的，再叠近的

`sort_gaussians` 按 z 深度从大到小排列；`rasterize_with_blending` 对每个覆盖像素执行颜色更新。

读代码时，把“排序得到的顺序”和“像素更新公式”对起来，就能看清遮挡是怎样产生的。

In [ ]:
class Gaussian:
    def __init__(self, position, scale, rotation, color, opacity):
        self.position = np.array(position) # World Space (x, y, z)
        self.scale = np.array(scale)
        self.rotation = np.array(rotation) # Euler angles
        self.color = np.array(color)
        self.opacity = opacity


def sort_gaussians(gaussians, view_matrix):
    """
    对高斯球进行深度排序 (从近到远? 还是从远到近?)
    提示：Alpha Blending 公式通常是从前向后(Front-to-Back)或从后向前(Back-to-Front)
    这里我们采用标准的 Painter's Algorithm (画家算法)，即从远到近画。
    """
    # 1. 计算每个高斯球在相机坐标系下的 Z 值 (Depth)
    # World Pos -> Camera Pos: P_cam = View_Matrix * P_world

    dtype = [('index', int), ('depth', float)]
    depths = []

    for i, g in enumerate(gaussians):
        # 简单的矩阵乘法: P_cam = R * P_world + t (这里简化为只有旋转，假设相机在原点)
        # 实际项目中 View Matrix 是 4x4，包含平移
        # 这里为了复用之前的代码，我们假设 view_matrix 只是 3x3 旋转
        # 且相机在原点，看向 Z 正方向

        pos_cam = view_matrix @ g.position
        depth = pos_cam[2] # Z value
        depths.append((i, depth))
    # 如果是从远到近画 (Back-to-Front)，应该按深度降序还是升序？
    # 提示：深度大 = 远
    # 使用 numpy 的 argsort 进行排序，然后反转得到降序
    # 或者直接用 Python 的 sort

    # 方法 1: Python sort
    depths.sort(key=lambda x: x[1], reverse=True)
    sorted_indices = [x[0] for x in depths]

    return sorted_indices


def render_scene(gaussians, view_matrix, focal_x, focal_y, width, height):
    canvas = np.zeros((height, width, 3))

    # 1. 排序
    sorted_indices = sort_gaussians(gaussians, view_matrix)

    # 2. 按顺序渲染
    for idx in sorted_indices:
        g = gaussians[idx]

        # --- 3D 部分 ---
        # 计算 3D 协方差
        R = get_rotation_matrix_from_euler(*g.rotation)
        cov3d = compute_covariance_3d(g.scale, R)

        # 投影到 2D
        pos_cam = view_matrix @ g.position
        cov2d = compute_2d_covariance(cov3d, view_matrix, focal_x, focal_y, pos_cam)

        # --- 2D 光栅化部分 (简化版，直接画在 canvas 上) ---
        # 注意：这里为了演示简单，我们还是用 simple_rasterizer 里的逻辑
        # 但实际上应该在这里实现 Alpha Blending
        # 为了不让代码太复杂，我们直接调用一个修改版的绘制函数
        rasterize_with_blending(canvas, g, cov2d, pos_cam, width, height, focal_x, focal_y)

    return canvas


def rasterize_with_blending(canvas, gaussian, cov2d, pos_cam, width, height, fx, fy):
    """
    将单个高斯混合到画布上 (Painter's Algorithm: 直接覆盖/混合)
    """
    # 计算屏幕中心坐标
    # u = fx * x / z + cx (假设 cx = width / 2)
    center_u = fx * pos_cam[0] / pos_cam[2] + width / 2
    center_v = fy * pos_cam[1] / pos_cam[2] + height / 2

    inv_cov = compute_inverted_covariance(cov2d)
    if inv_cov is None: return

    min_x, max_x, min_y, max_y = get_bounding_box(center_u, center_v, cov2d, width, height)

    for y in range(min_y, max_y):
        for x in range(min_x, max_x):
            delta = np.array([x - center_u, y - center_v])
            power = math.exp(-0.5 * (delta @ inv_cov @ delta))
            alpha = gaussian.opacity * power

            if alpha > 0.05:
                # Painter's Algorithm (Back-to-Front):
                # New_Color = Alpha * Src_Color + (1 - Alpha) * Dest_Color
                # 这就是标准的 Alpha Blending 公式

                pixel_color = canvas[y, x]
                new_pixel_color = alpha * gaussian.color + (1 - alpha) * pixel_color
                canvas[y, x] = new_pixel_color

In [ ]:
scene = [
    Gaussian([0, 0, 3], [0.35, 0.25, 0.10], [0, 0, 0], [1.0, 0.1, 0.1], 0.7),
    Gaussian([0, 0, 6], [0.70, 0.50, 0.20], [0, 0, 0], [0.1, 0.2, 1.0], 0.8),
]
view_rotation = np.eye(3)
focal_x = focal_y = 120.0
correct_order = sort_gaussians(scene, view_rotation)
rendered = render_scene(scene, view_rotation, focal_x, focal_y, width, height)
print("Back-to-front indices:", correct_order)
plt.figure(figsize=(6, 4))
plt.imshow(np.clip(rendered, 0, 1))
plt.title("Back-to-front alpha blending")
plt.show()

蓝色高斯较远，先落到黑色画布上；红色高斯较近，随后叠上去。红色中心 alpha 较高，遮住更多蓝色；红色边缘较淡，后面的蓝色就更容易透出来。

## 3. 只交换顺序，重叠区会变成什么颜色？

保持位置、大小和颜色都一样，只把绘制顺序反过来。这样两幅图的差别就来自“谁最后盖上去”。

下一格把正确顺序、反转顺序和逐像素绝对差并排摆出来。先猜猜：差异会集中在什么地方？

In [ ]:
reversed_canvas = np.zeros_like(rendered)
for index in reversed(correct_order):
    gaussian = scene[index]
    rotation = get_rotation_matrix_from_euler(*gaussian.rotation)
    covariance_3d = compute_covariance_3d(gaussian.scale, rotation)
    camera_center = view_rotation @ gaussian.position
    covariance_2d = compute_2d_covariance(covariance_3d, view_rotation, focal_x, focal_y, camera_center)
    rasterize_with_blending(
        reversed_canvas, gaussian, covariance_2d, camera_center,
        width, height, focal_x, focal_y,
    )
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
images = [rendered, reversed_canvas, np.abs(rendered - reversed_canvas)]
for ax, image, title in zip(axes, images, ["Far to near", "Near to far", "Absolute difference"]):
    ax.imshow(np.clip(image, 0, 1))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

重点看两个光斑的交叠区域：那里经历了两次混合，顺序改变会留下明显差别。只有一个高斯覆盖的地方，颜色基本保持原样。

## 现在可以解释一张重叠图了

<details>
<summary>高斯中心为什么亮、边缘为什么淡？</summary>

中心的高斯权重为 1，偏离中心后指数项逐渐变小。颜色贡献随 opacity × 高斯权重一起衰减。

</details>

<details>
<summary>加法混合和 Alpha 混合各在做什么？</summary>

加法混合把颜色贡献相加；Alpha 混合加入新颜色，同时按剩余透明度衰减旧画面。后者因此表现出遮挡和顺序关系。

</details>

<details>
<summary>把 alpha 调得更小，会看到什么？</summary>

当前层更淡，旧画面留下的比例更大，后面的颜色更容易透出来。

</details>

可以改红色高斯的 opacity，再观察交叠处的颜色。下一篇让参数自己调整：[可微渲染与图像拟合](03_可微渲染与图像拟合.ipynb)。